In [1]:
cd /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io

/work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io


In [2]:
datadir = '/work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/'


In [3]:
ls

CNAME     README.md  class2/  data/       js/        struct_trees/
LICENSE   about/     css/     fig/        pairwise/  tmp/
Makefile  class1/    d/       index.html  scripts/


In [9]:
import os
#run pdbfixer on all structs
from Bio import PDB
from pdbfixer import PDBFixer
from openmm.app import PDBFile
import glob
import sys
import tqdm
import pandas as pd
import multiprocessing as mp
#argument parser for the script 
from concurrent.futures import TimeoutError
from pebble import ProcessPool, ProcessExpired
import os
import glob
#list contents of directory
os.listdir('./')

['index.html',
 'LICENSE',
 'fig',
 'Makefile',
 'tmp',
 'class1',
 'd',
 'class2',
 'css',
 'CNAME',
 'pairwise',
 'README.md',
 '.gitignore',
 'data',
 'about',
 'struct_trees',
 '.git',
 'js',
 'scripts']

In [ ]:

import glob
structs = glob.glob( './class*/*/data/domains/Anticodon*/structures/*.pdb')
print(len(structs))
structs += glob.glob( './class*/*/data/domains/Catalytic*/structures/*.pdb')
print( len(structs) )



482
1030
['./class1/cys/data/domains/Anticodon_binding_domain_CRIMVL/structures/CysRS_AF_Arch_Aciduliprofundum_boonei_T469_gene8827372.pdb', './class1/cys/data/domains/Anticodon_binding_domain_CRIMVL/structures/CysRS_AF_Euk_Homo_sapiens_gene833.pdb', './class1/cys/data/domains/Anticodon_binding_domain_CRIMVL/structures/CysRS_AF_Bact_Streptobacillus_moniliformis_DSM_12112_gene29673427.pdb', './class1/cys/data/domains/Anticodon_binding_domain_CRIMVL/structures/CysRS_AF_Mito_Morone_saxatilis_gene118339815.pdb', './class1/cys/data/domains/Anticodon_binding_domain_CRIMVL/structures/CysRS_AF_Arch_Candidatus_Nitrosopumilus_sediminis_gene13696907.pdb', './class1/cys/data/domains/Anticodon_binding_domain_CRIMVL/structures/CysRS_AF_Bact_Mycoplasma_hyopneumoniae_168-L_gene57101512.pdb', './class1/cys/data/domains/Anticodon_binding_domain_CRIMVL/structures/CysRS_AF_Vir_Megavirus_chiliensis_gene11257271.pdb', './class1/cys/data/domains/Anticodon_binding_domain_CRIMVL/structures/CysRS_AF_Euk_Arabido

In [20]:
import pandas as pd

struct_df = { ( s.split('/')[5] , s.split('/')[-1].split('.')[0]):  { 'path': s , 'class': s.split('/')[1] , 'aa': s.split('/')[2] , 'type':s.split('/')[5] } for s in structs }
struct_df = pd.DataFrame.from_dict( struct_df , orient='index' )	
struct_df['group'] = struct_df['class']+'_' + struct_df['aa'] + '_'+struct_df['type']

print( struct_df.head()  , struct_df['type'].unique(),  len(struct_df) )

struct_df.to_csv( datadir + 'structs.csv' )


                                                                                                                                 path  \
Anticodon_binding_domain_CRIMVL CysRS_AF_Arch_Aciduliprofundum_boonei_T469_gene...  ./class1/cys/data/domains/Anticodon_binding_do...   
                                CysRS_AF_Euk_Homo_sapiens_gene833                   ./class1/cys/data/domains/Anticodon_binding_do...   
                                CysRS_AF_Bact_Streptobacillus_moniliformis_DSM_...  ./class1/cys/data/domains/Anticodon_binding_do...   
                                CysRS_AF_Mito_Morone_saxatilis_gene118339815        ./class1/cys/data/domains/Anticodon_binding_do...   
                                CysRS_AF_Arch_Candidatus_Nitrosopumilus_sedimin...  ./class1/cys/data/domains/Anticodon_binding_do...   

                                                                                     class  \
Anticodon_binding_domain_CRIMVL CysRS_AF_Arch_Aciduliprofundum_boonei_T469_gene... 

In [21]:
import shutil
clear = False
#setup a folder for each group in datadir
for group in struct_df['group'].unique():
	if clear == True:
		shutil.rmtree( datadir + group , ignore_errors=True )
	os.makedirs( datadir + group  , exist_ok=True)
	os.makedirs( datadir + group + '/structures'  , exist_ok=True)

In [ ]:
#copy the structures to the appropriate folder
for index, row in tqdm.tqdm(struct_df.iterrows()):
	shutil.copy( row['path'] , datadir + row['group'] + '/structures/' + index[1] + '.pdb' )

1026it [00:14, 72.24it/s]


In [24]:
#make directory for the fixed structures
for group in struct_df['group'].unique():
	os.makedirs(datadir+group+'/structs_fixed', exist_ok=True)


In [25]:
fixeq = glob.glob( datadir + '*/structures/*.pdb')
print( len(fixeq) )

1026


In [27]:
def prepchain( pdbfile , chain=None, savepath=None ,unid='',  verbose = False):
	assert savepath is not None
	
	try:
		#parse the pdb file
		parser = PDB.PDBParser()
		structure = parser.get_structure(unid, pdbfile)
		if chain:
			chainID = chain
		else:
			chainID = list(structure[0].get_chains())[0].get_id()
		chain_A = structure[0][chainID]
	except:
		print("error")
		return None
	if chain_A:
		io = PDB.PDBIO()
		io.set_structure(chain_A)
		io.save(savepath)
		fixer = PDBFixer(filename=savepath)
		fixer.findNonstandardResidues()
		fixer.replaceNonstandardResidues()
		fixer.removeHeterogens(True)
		fixer.findMissingResidues()
		fixer.findMissingAtoms()
		fixer.addMissingAtoms()
		#fixer.addMissingHydrogens(7.0)
		PDBFile.writeFile(fixer.topology, fixer.positions, open(savepath, 'w'))
		return None

fix_structs = True
if fix_structs:
	with ProcessPool() as pool:
		futures = [ pool.schedule( prepchain,  ( pdb , None , pdb.replace('structures', 'structs_fixed' ) , '' , False ) , timeout = 120) for pdb in fixeq  ]
		for future in tqdm.tqdm(futures, total=len(structs)):
			try:
				results = future.result()
			except TimeoutError as error:
				print("unstable_function took longer than %d seconds" % error.args[1])
			except ProcessExpired as error:
				print("%s. Exit code: %d" % (error, error.exitcode))
			except Exception as error:
				print("unstable_function raised %s" % error)
				print(error.traceback)  # Python's traceback of remote process
		pool.close()
		pool.join()



  0%|          | 0/1030 [00:00<?, ?it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 3279.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 3355.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 4072.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain C is discontinuous at line 3441.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuil

unstable_function took longer than 120 seconds


 63%|██████▎   | 645/1030 [07:30<28:52,  4.50s/it]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 1263.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 1463.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 1624.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 1743.
  warnings.warn(
 71%|███████   | 731/1030 [08:06<04:36,  1.08it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/e

In [28]:
import pandas as pd
import Bio.PDB
import numpy as np
import os
import tqdm

def extract_core(resdf , outfile,  hitthresh = .8 ,minthresh = .6, corefolder = 'core_structs/' , structfolder = 'structs/' , cterfolder = 'cter_structs/' , nterfolder = 'nter_structs/'  , folder = None):
	"""

	Extract a core of structures from a results file

	Parameters
		resdf: path to results file
		outfile: path to output file
		hitthresh: proportion of structures that need to map to a residue for it to be included in the core
		minthresh: if no residues meet the hitthresh, the minimum proportion of structures that need to map to a residue for it to be included in the core
		corefolder: name of folder to output core structures to
		structfolder: name of folder to find structures in
		cterfolder: name of folder to find cter structures in
		nterfolder: name of folder to find nter structures in


	"""
	
	#read all results
	if folder is None:
		folder =''.join([ sub + '/' for sub in resdf.split('/')[:-1] ])
	
	print(folder)
	resdf = pd.read_table(resdf, header = None)
	resdf.columns = 'query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,lddt,lddtfull,alntmscore'.split(',')

	print('extracting core')
	print('hitthresh: ' + str(hitthresh))
	print('minthresh: ' + str(minthresh))

	print(resdf.head(), len(resdf['query'].unique()))
	#map hits to each struc
	hits = {}
	nqueries = len(resdf['query'].unique())
	#proportion of structures that need to map to a residue fo
	with tqdm.tqdm(total=len(resdf['query'].unique())) as pbar:
		for i,q in enumerate(resdf['query'].unique()):
			sub = resdf[resdf['query'] == q]
			hitvec = np.zeros((1,max(sub['qend'])) )
			for idx,r in sub.iterrows():
				hitvec[0,r['qstart']:r['qend']] = hitvec[0,r['qstart']:r['qend']]+1
			hitvec /= nqueries
			core = np.where(hitvec>hitthresh)[1]
			try:
				hits[q]= { 'min': np.amin(core), 'max': np.amax(core) }
			except:
				#be more lenient...
				subthresh = np.amax(hitvec)
				print(hitvec, sub, 'be careful, non homologous sequences may have enterred the dataset!')
				if subthresh>=minthresh:
					print('new core threst set at ' + str(subthresh) )
					core = np.where(hitvec>=subthresh)[1]
					hits[q]= { 'min': np.amin(core), 'max': np.amax(core)}
					print(q , 'added')
				else:
					print(q , 'rejected')
			pbar.set_description('processed: %d' % (1 + i))
			pbar.update(1)
	#make core struct folder
	os.makedirs(folder+corefolder , exist_ok = True)
	os.makedirs(folder+cterfolder , exist_ok = True)
	os.makedirs(folder+nterfolder , exist_ok = True)
	#parse each struct and output core to folder 
	parser = Bio.PDB.PDBParser()
	with tqdm.tqdm(total=len(hits)) as pbar:
		for i,q in enumerate(hits):
			if not q.endswith('.pdb'):
				qfile = q + '.pdb'
			else:
				qfile = q
			if '/' in q:
				qfile = qfile.split('/')[-1]

			struct = parser.get_structure(q.split('.')[0], structfolder+qfile )

			#zero based indexing...
			struct_core = Bio.PDB.Dice.extract( struct ,'A' , hits[q]['min']+1 , hits[q]['max']+1 ,folder+corefolder+qfile  )
			
			struct_core = Bio.PDB.Dice.extract( struct ,'A' , 0, hits[q]['min']+1  ,folder+nterfolder+q  )
			#select from max to end
			struct_core = Bio.PDB.Dice.extract( struct ,'A' , hits[q]['max']+1 , len(struct[0]['A'])  ,folder+cterfolder+qfile  )

			hits[q]['len'] = [ len(chain) for model in struct for chain in model ][0]

			pbar.set_description('processed: %d' % (1 + i))
			pbar.update(1)


	hitsdf = pd.DataFrame.from_dict( hits , orient='index'  )
	hitsdf.to_csv(outfile)
	return folder +'struct_cores.csv'




In [29]:
import sys
import os

sys.path.append('../../snake_tree/src')
import numpy as np
import pandas as pd
import foldseek2tree


In [30]:

def structblob2tree(input_folder, outfolder, overwrite = False,
			 fastmepath = 'fastme', quicktreepath = 'quicktree' , 
		 foldseekpath = 'foldseek' , delta = 0.0001 ,
	   correction = False , kernels = ['fident' , 'lddt' , 'alntmscore' ] , core = False 
	   , hittresh = .8 , minthresh = .6 , swapids = False):
	'''
	run fold tree pipeline for a folder of pdb files
	
	Parameters
	----------
	input_folder : str
		path to folder with pdb files   
	outfolder : str
		path to output folder   
	overwrite : bool
		overwrite existing foldseek output  
	fastmepath : str    
		path to fastme executable
	quicktreepath : str 
		path to quicktree executable
	foldseekpath : str  
		path to foldseek executable 
	delta : float   
		small number to replace negative branch lengths with, default is .0001
	correction : str    
		correction method to use, either 'tajima' or 'none'
	kernel : str    
		kernel to use, either 'fident', 'lddt' or 'alntmscore'
	
	'''
	#check if the foldseek output is already there
	if os.path.exists(outfolder + 'res.m8') and overwrite == False:
		print('found foldseek output, skipping foldseek')
		alnres = outfolder + 'res.m8'
	else:
		alnres = foldseek2tree.runFoldseek_allvall_EZsearch(input_folder , outfolder + 'res.m8', foldseekpath = foldseekpath)
	
	if core == True:
		core_designator = 'core'
		extract_core( alnres , alnres+'.core.csv',  hitthresh = .8 ,minthresh = .6  , structfolder = input_folder)
		corefolder = ''.join([ sub + '/' for sub in alnres.split('/')[:-1] ])+'core_structs/'
		if os.path.exists(outfolder + 'core.res.m8') and overwrite == False:
			print('found foldseek core output, skipping foldseek')
			alnres = outfolder + 'core.res.m8'
		else:
			alnres = foldseek2tree.runFoldseek_allvall_EZsearch(corefolder  , outfolder + 'core.res.m8', foldseekpath = foldseekpath  )
	else:
		core_designator = ''
	
	res = pd.read_table(alnres , header = None )
	res[0] = res[0].map(lambda x :x.replace('.pdb', ''))
	res[1] = res[1].map(lambda x :x.replace('.pdb', ''))
	res.columns = 'query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,lddt,lddtfull,alntmscore'.split(',')
	ids = sorted(list( set(list(res['query'].unique()) + list(res['target'].unique()))))
	if swapids:
		res['numerical_query'] = res['query'].map(lambda x : ids.index(x))
		res['numerical_target'] = res['target'].map(lambda x : ids.index(x))
	
	pos = { protid : i for i,protid in enumerate(ids)}
	if swapids:
		ids = [ str(i) for i in range(len(ids))]

	matrices = { kernel:np.zeros((len(pos), len(pos))) for kernel in kernels} 

	#cast kernel columns to float
	for k in kernels:
		res[k] = res[k].astype(float )

	#calc kernel for tm, aln score, lddt
	for idx,row in res.iterrows():
		for k in matrices:
			matrices[k][pos[row['query']] , pos[row['target']]] += row[k]
			matrices[k][pos[row['target']] , pos[row['query']]] += row[k]
	trees = {}
	for i,k in enumerate(matrices):
		matrices[k] /= 2
		matrices[k] = 1-matrices[k]
		matrices[k] = np.clip(matrices[k], 0, 1)
		
		print(matrices[k], np.amax(matrices[k]), np.amin(matrices[k]) )
		if correction:
			if k == 'fident':
				factor = .93
			else:
				factor = 1
			matrices[k] = foldseek2tree.Tajima_dist(matrices[k], bfactor = factor)
		np.save( input_folder + k + core_designator + '_distmat.npy' , matrices[k])
		distmat_txt = foldseek2tree.distmat_to_txt( ids , matrices[k] , outfolder + k + core_designator + '_distmat.txt' )
		out_tree = foldseek2tree.runFastme(  fastmepath = fastmepath , clusterfile = distmat_txt )
		out_tree = foldseek2tree.postprocess(out_tree, input_folder + k + core_designator + 'structblob_tree.nwk' , delta = delta)
		trees[k] = out_tree
	return res, trees

In [32]:
#make corecut dir in each group

for group in struct_df['group'].unique():
	os.makedirs(datadir+group+'/core_structs', exist_ok=True)
	os.makedirs(datadir+group+'/ft1tree', exist_ok=True)

results = {}
for folder in struct_df['group'].unique():
	#foldtree basic
	alnres, trees = structblob2tree(input_folder = datadir+folder +'/structs_fixed/', outfolder = datadir+folder+'/ft1tree/' , overwrite = False,
				fastmepath = 'fastme', quicktreepath = 'quicktree' , 
			foldseekpath = 'foldseek' , delta = 0.0001 ,
		correction = True ,  core = False 
		, hittresh = .8 , minthresh = .6  , swapids=True) 

	alnres_core , trees_core = structblob2tree(input_folder = datadir+folder +'/structs_fixed/' , outfolder = datadir+folder+'/ft1tree/' , overwrite = False,
				fastmepath = 'fastme', quicktreepath = 'quicktree' , 
			foldseekpath = 'foldseek' , delta = 0.0001 ,
		correction = True ,  core = True 
		, hittresh = .8 , minthresh = .6 , swapids=True)
	results[folder] = { 'alnres': alnres, 'trees': trees , 'alnres_core': alnres_core , 'trees_core': trees_core }

easy-search /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/class1_cys_Anticodon_binding_domain_CRIMVL/structs_fixed/ /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/class1_cys_Anticodon_binding_domain_CRIMVL/structs_fixed/ /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/class1_cys_Anticodon_binding_domain_CRIMVL/ft1tree/res.m8 tmp --format-output query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,lddt,lddtfull,alntmscore --exhaustive-search 

MMseqs Version:              	9.427df8a
Seq. id. threshold           	0
Coverage threshold           	0
Coverage mode                	0
Max reject                   	2147483647
Max accept                   	2147483647
Add backtrace                	false
TMscore threshold            	0
TMalign hit order            	0
TMalign fast                 	1
Preload mode                 	0
Threads              

  0%|          | 0/23 [00:00<?, ?it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 1533
  warnings.warn(
processed: 1:   0%|          | 0/23 [00:00<?, ?it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 1734
  warnings.warn(
processed: 2:   4%|▍         | 1/23 [00:00<00:00, 33.76it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 1656
  warnings.warn(
processed: 3:   9%|▊         | 2/23 [00:00<00:00, 44.99it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'E

easy-search /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/class1_cys_Anticodon_binding_domain_CRIMVL/ft1tree/core_structs/ /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/class1_cys_Anticodon_binding_domain_CRIMVL/ft1tree/core_structs/ /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/class1_cys_Anticodon_binding_domain_CRIMVL/ft1tree/core.res.m8 tmp --format-output query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,lddt,lddtfull,alntmscore --exhaustive-search 

MMseqs Version:              	9.427df8a
Seq. id. threshold           	0
Coverage threshold           	0
Coverage mode                	0
Max reject                   	2147483647
Max accept                   	2147483647
Add backtrace                	false
TMscore threshold            	0
TMalign hit order            	0
TMalign fast                 	1
Preload mode                 	0
Th

processed: 23: 100%|██████████| 23/23 [00:00<00:00, 400.04it/s]


[[0.         0.08695652 0.26086957 0.26086957 0.26086957 0.26086957
  0.26086957 0.26086957 0.34782609 0.34782609 0.34782609 0.34782609
  0.34782609 0.34782609 0.34782609 0.39130435 0.39130435 0.39130435
  0.39130435 0.39130435 0.39130435 0.39130435 0.39130435 0.43478261
  0.52173913 0.56521739 0.56521739 0.60869565 0.69565217 0.69565217
  0.69565217 0.65217391 0.65217391 0.65217391 0.65217391 0.69565217
  0.69565217 0.69565217 0.69565217 0.73913043 0.73913043 0.73913043
  0.73913043 0.52173913 0.47826087 0.47826087 0.34782609 0.34782609
  0.34782609 0.34782609 0.26086957 0.26086957 0.26086957 0.26086957
  0.26086957 0.26086957 0.26086957 0.26086957 0.26086957 0.26086957
  0.26086957 0.26086957 0.26086957 0.26086957 0.26086957 0.26086957
  0.26086957 0.26086957 0.26086957 0.26086957 0.08695652 0.08695652
  0.04347826 0.08695652 0.26086957 0.26086957 0.26086957 0.26086957
  0.26086957 0.26086957 0.26086957 0.26086957 0.26086957 0.26086957
  0.26086957 0.26086957 0.26086957 0.2173913  0.

...

easy-search /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/class2_phe3_Catalytic_domain/ft1tree/core_structs/ /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/class2_phe3_Catalytic_domain/ft1tree/core_structs/ /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/class2_phe3_Catalytic_domain/ft1tree/core.res.m8 tmp --format-output query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,lddt,lddtfull,alntmscore --exhaustive-search 

MMseqs Version:              	9.427df8a
Seq. id. threshold           	0
Coverage threshold           	0
Coverage mode                	0
Max reject                   	2147483647
Max accept                   	2147483647
Add backtrace                	false
TMscore threshold            	0
TMalign hit order            	0
TMalign fast                 	1
Preload mode                 	0
Threads                      	48
Verbosity  

  0%|          | 0/9 [00:00<?, ?it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 3008
  warnings.warn(
processed: 1:   0%|          | 0/9 [00:00<?, ?it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 3251
  warnings.warn(
processed: 2:  11%|█         | 1/9 [00:00<00:00, 18.47it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 3486
  warnings.warn(
processed: 3:  22%|██▏       | 2/9 [00:00<00:00, 24.05it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' 

easy-search /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/class2_phe4_Catalytic_domain/ft1tree/core_structs/ /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/class2_phe4_Catalytic_domain/ft1tree/core_structs/ /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/class2_phe4_Catalytic_domain/ft1tree/core.res.m8 tmp --format-output query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,lddt,lddtfull,alntmscore --exhaustive-search 

MMseqs Version:              	9.427df8a
Seq. id. threshold           	0
Coverage threshold           	0
Coverage mode                	0
Max reject                   	2147483647
Max accept                   	2147483647
Add backtrace                	false
TMscore threshold            	0
TMalign hit order            	0
TMalign fast                 	1
Preload mode                 	0
Threads                      	48
Verbosity  

  0%|          | 0/7 [00:00<?, ?it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 3907
  warnings.warn(
processed: 1:   0%|          | 0/7 [00:00<?, ?it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 4348
  warnings.warn(
processed: 2:  14%|█▍        | 1/7 [00:00<00:00, 15.44it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 4946
  warnings.warn(
processed: 3:  43%|████▎     | 3/7 [00:00<00:00, 28.77it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' 

easy-search /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/class2_phe5_Catalytic_domain/ft1tree/core_structs/ /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/class2_phe5_Catalytic_domain/ft1tree/core_structs/ /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/class2_phe5_Catalytic_domain/ft1tree/core.res.m8 tmp --format-output query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,lddt,lddtfull,alntmscore --exhaustive-search 

MMseqs Version:              	9.427df8a
Seq. id. threshold           	0
Coverage threshold           	0
Coverage mode                	0
Max reject                   	2147483647
Max accept                   	2147483647
Add backtrace                	false
TMscore threshold            	0
TMalign hit order            	0
TMalign fast                 	1
Preload mode                 	0
Threads                      	48
Verbosity  

In [35]:
#save the dataframes 
for folder in results:
	for key in results[folder]:
		if key.startswith('alnres'):
			results[folder][key].to_csv(datadir+folder+'/ft1tree/' + key + '.csv')



In [36]:

#reformat the tree files with the name mapper
import ete3


In [37]:

for folder in results:
	for key in results[folder]:
		df = results[folder]['alnres']
		mapper = dict( zip( df['numerical_query'] , df['query'] ))
		if key.startswith('trees'):
			for k in results[folder][key]:
				t = ete3.Tree(results[folder][key][k])
				print(k , results[folder][key][k])
				for leaf in t:
					leaf.name = mapper[int(leaf.name)]
				print(t)
				t.write(outfile =results[folder][key][k]+'.rename.nwk', format = 0 )


fident /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_trees/class1_cys_Anticodon_binding_domain_CRIMVL/structs_fixed/fidentstructblob_tree.nwk

   /-CysRS_AF_Bact_Mycobacterium_canettii_CIPT_140010059_gene45427568
  |
  |--CysRS_PDB_Bact_M_smegmatis_3C8Z
  |
  |   /-CysRS_AF_Bact_Treponema_pallidum_subsp_pertenue_str_SamoaD_gene57878631
--|  |
  |  |      /-CysRS_AF_Arch_Pyrobaculum_ferrireducens_gene11593896
  |  |     |
  |  |   /-|      /-CysRS_AF_Mito_Morone_saxatilis_gene118339815
  |  |  |  |   /-|
   \-|  |  |  |   \-CysRS_AF_Mito_Homo_sapiens_gene79587
     |  |   \-|
     |  |     |   /-CysRS_AF_Euk_Cyanidioschyzon_merolae_strain_10D_gene16993418
     |  |      \-|
     |  |        |   /-CysRS_AF_Mito_Saccharomyces_cerevisiae_S288C_gene855474
     |  |         \-|
      \-|           |   /-CysRS_AF_Euk_Homo_sapiens_gene833
        |            \-|
        |               \-CysRS_AF_Euk_Drosophila_melanogaster_gene36784
        |
        |    

NewickError: Unexisting tree file or Malformed newick tree structure.
You may want to check other newick loading flags like 'format' or 'quoted_node_names'.

In [ ]:
import shutil
#copy the cores and set things up for the snakemake pipeline


os.makedirs(datadir+'mltrees/', exist_ok=True)
for fam in ['c1', 'c2', 'CRIMVLG']:
	input_folder = datadir+folder +'/ft1tree/'
	cpdir = input_folder+'core_structs/'
	os.makedirs(datadir+'mltrees/'+fam, exist_ok=True)
	os.makedirs(datadir+'mltrees/'+fam+'/structs', exist_ok=True)
	#copy the core structures
	for f in glob.glob(cpdir+'/*'):
		#only copy the pdb files if they are not already there
		if not os.path.exists(datadir+'mltrees/'+fam+'/structs/'+f.split('/')[-1]):
			shutil.copy(f, datadir+'mltrees/'+fam+'/structs/')
	with open( datadir+'mltrees/'+fam+'/identifiers.txt' , 'w') as f:
		for i in range(len(glob.glob(cpdir+'/*'))):
			f.write(str(i)+'\n')

	#write a dummy finalset.csv file to each folder
	with open( datadir+'mltrees/'+fam+'/finalset.csv' , 'w') as f:
		f.write('pdb,chain,uniprot\n')
		f.write('dummy,dummy,dummy\n')
	